# Atelier 2.D — Exposer l'assistant en API ★

**Durée : 30 min** · Projet fil rouge, incrément du Jour 2

Un notebook ne s'intègre à rien. Vous exposez l'assistant documentaire comme un **service web** (FastAPI), que l'agent du Jour 4 et le workflow du Jour 5 appelleront.

Le service est le **backend de l'application** du module (`../application/backend`) : il construit l'assistant au démarrage, avec le pipeline des ateliers (`rag.py`), et expose cinq routes. Le frontend Next.js de l'application appelle ces mêmes routes.

| Route | Rôle |
|---|---|
| `GET /api/sante` | état du service : modèle, documents, extraits, reranking, LangSmith |
| `GET /api/documents` | documents publics interrogés |
| `GET /api/exemples` | une question du golden set par type |
| `POST /api/recherche` | extraits retrouvés, **sans appel au modèle** |
| `POST /api/question` | réponse citée, extraits transmis, sources, latence, coût estimé |

In [ ]:
from pathlib import Path

BACKEND = Path("../application/backend").resolve()
print((BACKEND / "app" / "main.py").read_text(encoding="utf-8"))

**À observer.**

- Les schémas Pydantic (`app/schemas.py`) décrivent le **contrat** du service : FastAPI valide les entrées, documente l'API et refuse une question vide ou trop longue.
- L'index est construit **une seule fois**, au démarrage (`cycle_de_vie`), et non à chaque requête.
- `creer_application` reçoit la fabrique de l'assistant en paramètre : les tests (`backend/tests`) la remplacent par un assistant léger, sans modèle à télécharger ni clé.

In [ ]:
print((BACKEND / "app" / "schemas.py").read_text(encoding="utf-8"))

## 1. Démarrer le service

On lance `uvicorn` en arrière-plan, depuis le dossier du backend. Le démarrage prend quelques secondes : chargement des modèles d'embeddings et de reranking, construction de l'index. Le service lit le même `.env` que les ateliers ; on lui indique le cache des modèles déjà téléchargés par les ateliers 2.A à 2.C.

In [ ]:
import os
import subprocess
import sys
import time

import httpx

PORT = 8000
URL = f"http://127.0.0.1:{PORT}"
environnement = {**os.environ, "CACHE_MODELES": os.getenv("CACHE_MODELES", str(Path(".cache_modeles").resolve()))}
journal = open("serveur.log", "w")
serveur = subprocess.Popen([sys.executable, "-m", "uvicorn", "app.main:app", "--port", str(PORT)],
                           cwd=BACKEND, env=environnement, stdout=journal, stderr=subprocess.STDOUT)

for _ in range(180):
    try:
        if httpx.get(f"{URL}/api/sante", timeout=2).status_code == 200:
            break
    except httpx.TransportError:
        time.sleep(1)
print(httpx.get(f"{URL}/api/sante").json())

La documentation interactive est disponible sur http://127.0.0.1:8000/docs : ouvrez-la dans votre navigateur et testez les routes.

## 2. Interroger le service

In [ ]:
import json

print(json.dumps(httpx.get(f"{URL}/api/documents").json(), ensure_ascii=False, indent=2)[:600])

In [ ]:
reponse = httpx.post(f"{URL}/api/question", json={"question": "Livrez-vous en France ?"}, timeout=180)
print(reponse.status_code)
corps = reponse.json()
print(corps["reponse"], "\n")
print("Sources :", corps["sources"], "| modèle :", corps["modele"], "| latence :", corps["latence_s"], "s")
for extrait in corps["extraits"]:
    print(f"  [{extrait['numero']}] {'cité ' if extrait['cite'] else '     '} {extrait['document']} ({extrait['fichier']})")

Le même appel depuis un terminal :

```bash
curl -X POST http://127.0.0.1:8000/api/question \
     -H "Content-Type: application/json" \
     -d '{"question": "Livrez-vous en France ?"}'
```

La route `/api/recherche` montre ce que trouve la recherche, **sans appeler le modèle** : utile pour comprendre une mauvaise réponse.

In [ ]:
resultat = httpx.post(f"{URL}/api/recherche", json={"question": "Quelle remise pour une grosse commande ?", "k": 3}).json()
for extrait in resultat["extraits"]:
    print(f"[{extrait['numero']}] score {extrait['score']} | {extrait['document']} ({extrait['fichier']})")

Aucun extrait de la procédure interne : le filtre sur les documents publics s'applique aussi dans le service.

## 3. Une entrée invalide

FastAPI refuse la requête avant même d'appeler l'assistant, avec un message d'erreur explicite (code 422).

In [ ]:
reponse = httpx.post(f"{URL}/api/question", json={"question": ""}, timeout=30)
print(reponse.status_code)
print(json.dumps(reponse.json(), ensure_ascii=False, indent=2))

## 4. Plusieurs clients à la fois

In [ ]:
from concurrent.futures import ThreadPoolExecutor

questions = ["Quels sont les horaires du service client ?", "Les toners sont-ils couverts par la garantie ?",
             "Combien coûte la livraison express ?"]

def interroger(question):
    debut = time.perf_counter()
    r = httpx.post(f"{URL}/api/question", json={"question": question}, timeout=300).json()
    return question, r["reponse"][:120], round(time.perf_counter() - debut, 1)

with ThreadPoolExecutor(max_workers=3) as pool:
    for question, texte, duree in pool.map(interroger, questions):
        print(f"{duree:5.1f} s | {question}\n        {texte}…\n")

## 5. Arrêter le service

In [ ]:
serveur.terminate()
serveur.wait()
journal.close()
print(Path("serveur.log").read_text(encoding="utf-8")[-800:])

## L'application complète

Le backend s'utilise aussi hors du notebook, avec son interface web :

```bash
# terminal 1 : le service, depuis Module 02 - RAG/application/backend
uvicorn app.main:app --reload --port 8000

# terminal 2 : l'interface, depuis Module 02 - RAG/application/frontend
npm install && npm run dev       # http://localhost:3000
```

Le dossier `application/` contient aussi des tests (`pytest`, avec un LLM factice), un script d'évaluation sur le golden set (`python evaluer.py --seuil 0.85`) et des traces LangSmith en option. Voir `application/README.md`.

## Pour aller plus loin

- ✏️ Ajoutez une route `GET /api/golden-set` qui renvoie le nombre de questions du golden set par type.
- Avant une mise en production, il manquerait : l'authentification des appels, la limitation du débit, la journalisation des questions conforme à la politique de données (conservation 30 jours), et une mise à jour de l'index quand un document change.

## À retenir

- Un service web transforme le prototype en composant réutilisable : un contrat d'entrée et de sortie, une documentation, des erreurs explicites.
- L'index se construit au démarrage, pas à chaque requête.
- Ce service devient un **outil** de l'agent au Jour 4, et une étape du workflow RPA 2.0 au Jour 5.